In [1]:
from pathlib import Path

env_path = Path("../.env")
contenido = env_path.read_bytes()
print("Primeros 20 bytes:", contenido[:20])
print("Primeros 20 bytes hex:", contenido[:20].hex())

Primeros 20 bytes: b'PG_USER=aprende\nPG_P'
Primeros 20 bytes hex: 50475f555345523d617072656e64650a50475f50


In [2]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

print("PG_USER:", os.getenv('PG_USER'))
print("PG_HOST:", os.getenv('PG_HOST'))
print("PG_DB:", os.getenv('PG_DB'))

PG_USER: aprende
PG_HOST: localhost
PG_DB: aprende_rag


In [6]:
import psycopg

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="aprende_rag",
    user="aprende"
)

cur = conn.cursor()
cur.execute("SELECT version()")
print("✅ PostgreSQL conectado:", cur.fetchone()[0][:40])

✅ PostgreSQL conectado: PostgreSQL 16.14 (Debian 16.14-1.pgdg13+


In [7]:
cur = conn.cursor()
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""")
tablas = cur.fetchall()
print("Tablas creadas:")
for t in tablas:
    print(" -", t[0])

cur.close()
conn.close()

Tablas creadas:
 - documento_vectorizado
 - instrumento_procesado
 - kpi
 - pregunta_kpi
 - prompt
 - rag_log


In [9]:
import chromadb

client = chromadb.PersistentClient(path="../../data/chroma")

colecciones = ["col_encuestas", "col_entrevistas", "col_pruebas_estandarizadas"]
for nombre in colecciones:
    client.get_or_create_collection(name=nombre)

print("✅ ChromaDB conectado")
print("Colecciones:")
for col in client.list_collections():
    print(" -", col.name)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ ChromaDB conectado
Colecciones:
 - col_pruebas_estandarizadas
 - col_encuestas
 - col_entrevistas


In [15]:
import httpx

ollama_url = "http://localhost:11434"

r = httpx.get(ollama_url)
print("✅ Ollama corriendo:", r.text)

r2 = httpx.get(f"{ollama_url}/api/tags")
print("Response completo:", r2.json())
modelos = [m["name"] for m in r2.json()["models"]]
print("\nModelos disponibles:")
for m in modelos:
    print(" -", m)

✅ Ollama corriendo: Ollama is running
Response completo: {'models': [{'name': 'llama3.2:3b', 'model': 'llama3.2:3b', 'modified_at': '2026-06-22T00:44:35.5963169-06:00', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}, {'name': 'llama3.2:latest', 'model': 'llama3.2:latest', 'modified_at': '2026-05-04T15:35:28.1885913-06:00', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}, {'name': 'vicuna:latest', 'model': 'v

In [16]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
vector = modelo.encode("prueba de conexión educativa")

print("✅ Modelo de embeddings cargado")
print("Dimensión del vector:", vector.shape)

C:\proyectos\clean_standarization_RAG\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Modelo de embeddings cargado
Dimensión del vector: (768,)


In [17]:
import httpx

r = httpx.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Responde solo con 'OK': ¿estás funcionando?",
        "stream": False
    },
    timeout=60.0
)

respuesta = r.json()["response"]
print("✅ LLM responde:", respuesta)

✅ LLM responde: OK
